### Install all required Libraries

In [ ]:
!pip install datasets pandas numpy tqdm python-dotenv requests tqdm -q

In [ ]:
!pip install openai==0.28

In [ ]:
!pip install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.8/786.8 kB 17.9 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.99.8
    Uninstalling openai-1.99.8:
      Successfully uninstalled openai-1.99.8


### Import required Libraries

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd
import numpy as np
import os
from huggingface_hub import login
import getpass
from google.colab import userdata
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import gc
import warnings
warnings.filterwarnings('ignore')

import time
from matplotlib import pyplot as plt
import seaborn as sns
import ast
from datetime import datetime
import uuid
import os, json, math
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from openai import OpenAI
import time
from datasets.utils.logging import disable_progress_bar, set_verbosity_error
from datasets import load_dataset, concatenate_datasets, disable_caching

### Hugginface Account Login

In [ ]:
HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in using environment variable")
else:
    print("No HF_TOKEN found")

Logged in using environment variable


### Set Open AI API env variables

In [ ]:
from openai import OpenAI
import os

# OpenAI API key
api_key = userdata.get('OPENAI_API_KEY')

os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI(api_key=api_key)

### Display all the required columns and rows

In [ ]:
pd.set_option('display.max_rows', None)        # Show all rows
pd.set_option('display.max_columns', None)     # Show all columns
pd.set_option('display.width', None)           # Don't wrap to next line
pd.set_option('display.max_colwidth', None)    # Show full column content

In [ ]:
# Turn off HF progress bars & caching (avoids hashing warnings)
disable_progress_bar()
disable_caching()

### Mount Google Drive

In [ ]:
from google.colab import drive

# mount drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset_name   = "Lakshan2003/customer_service_200k_client_agent_conversations"
split          = "validation"                 # which dataset split to use
text_col       = "history_string"        # column containing the conversation text
id_col         = "conversation_id"       # unique ID for each conversation

BATCH_SIZE     = 100                       # number of rows to process per batch
MAX_ROWS       = None                    # None → whole split; int → first N rows of the ORIGINAL dataset
NUM_PROC       = 2                        # number of processes for mapping (set >1 to speed up)

# Output paths
out_csv        = f"/content/drive/MyDrive/fyp-2025/Datasets/Summaries/{split}/history_summaries.csv"
tmp_csv        = out_csv + ".tmp"         # temp file for atomic writes

# OpenAI API settings
MODEL              = "gpt-4o-mini"
TEMPERATURE        = 0.3
MAX_OUTPUT_TOKENS  = 250                   # 90–140 words output

# CSV columns to keep in final output
COLUMNS_TO_SAVE = [id_col, text_col, "summary", "orig_idx"]

### Helper functions

In [ ]:
def fmt_eta(seconds: float) -> str:
    """Convert seconds to HH:MM:SS for ETA display."""
    seconds = max(0, int(seconds))   # Ensure seconds is a non-negative integer

    # Break down seconds into hours, minutes, and remaining seconds.
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60

    # Return a zero-padded time string like "01:23:45".
    return f"{h:02d}:{m:02d}:{s:02d}"

def normalize_for_csv(ds_flat):
    """
    Keep only required columns & coerce to strings for CSV compatibility.
    Ensures consistent schema for merges/resume.
    """

    # Only keep the columns to  be saved in the csv
    keep = [c for c in COLUMNS_TO_SAVE if c in ds_flat.column_names]
    ds_sel = ds_flat.select_columns(keep)

     # Convert each batch of data into string-safe values.
    def to_str(batch):
        # IDs: empty string if missing.
        ids   = ["" if v is None else str(v) for v in batch[id_col]]

        # Texts: handle None and NaN values
        texts = ["" if (v is None or (isinstance(v, float) and math.isnan(v))) else str(v) for v in batch[text_col]]

        # Summaries: only include if "summary" column exists, else empty string.
        sums  = ["" if ("summary" not in batch or v is None) else str(v) for v in batch.get("summary", [""]*len(ids))]
        idxs  = batch.get("orig_idx", list(range(len(ids))))

         # Keep original index if available, otherwise generate a fallback index.
        return {id_col: ids, text_col: texts, "summary": sums, "orig_idx": idxs}

    # Apply conversion across the dataset without caching
    return ds_sel.map(to_str, batched=True, load_from_cache_file=False)

def atomic_write_csv(ds_to_write, dst_path, tmp_path):
    """
    Save dataset to a temp file first, then atomically replace the main CSV.
    This prevents Google Drive from creating duplicates on each run.
    """
    # Write CSV to temporary path first.
    ds_to_write.to_csv(tmp_path)

    # Move the temp file into place, replacing the old one safely.
    os.replace(tmp_path, dst_path)

### Load the Dataset

In [ ]:
# Load full dataset and tag each row with its original index for order preservation
ds_full = load_dataset(dataset_name)[split]
ds_full = ds_full.map(lambda ex, i: {"orig_idx": i}, with_indices=True, load_from_cache_file=False)

# If MAX_ROWS is set, only keep the first N rows while preserving order
if isinstance(MAX_ROWS, int):
    ds_target = ds_full.select(range(min(MAX_ROWS, len(ds_full))))
else:
    ds_target = ds_full

## Resume Logic
# Check if an output CSV already exists.
# If it does, read processed IDs and skip those rows to avoid duplicates.
if os.path.exists(out_csv):
    existing = load_dataset("csv", data_files=out_csv)["train"]
    done_ids = set(map(str, existing[id_col]))
    ds_remaining = ds_target.filter(lambda x: str(x[id_col]) not in done_ids)
else:
    # If no output CSV exists, start fresh with the full target set
    ds_remaining = ds_target

# Track how many rows are in the target window and how many remain
total_target = len(ds_target)
total_to_do  = len(ds_remaining)
print(f"[info] Target window: {total_target} rows (MAX_ROWS={MAX_ROWS if MAX_ROWS else 'ALL'})")
print(f"[info] Remaining: {total_to_do} rows to process")

# Stop execution if everything in the target set has already been processed
if total_to_do == 0:
    raise SystemExit("Nothing left to process in the selected window. All done.")
ds_full = load_dataset(dataset_name)[split]
ds_full = ds_full.map(lambda ex, i: {"orig_idx": i}, with_indices=True, load_from_cache_file=False)


[info] Target window: 18334 rows (MAX_ROWS=ALL)
[info] Remaining: 1634 rows to process


### Summarizing prompt

In [ ]:
SUMMARY_PROMPT = """
You are a professional conversation summarization assistant.

Goal: Produce a clear, concise, factual summary of the conversation so far so that YOU, the same customer service agent handling this client, can accurately answer their next question.

Include only information explicitly stated:
- Client’s issue/request and current status (who is the client and agent should be specifically mentioned if there names exist in the converstion)
- Verification steps completed or pending
- Exact names, accounts/identifiers, dates, amounts, and actions taken or agreed
- Commitments, deadlines, follow-ups
- Current status of the conversation

Exclude: greetings, filler, speculation, or invented details.

Style: neutral and professional. Vary sentence structure and phrasing to avoid repetition.

Output: one coherent detailed paragraph summary.

Conversation so far:
{conversation_data}
"""

### Open AI batch calling Map Function

In [ ]:
# Summarisation map function (picklable; stops on first error)
def hf_openai_summarise(batch, model: str, temperature: float, max_output_tokens: int,
                        prompt_template: str, text_key: str):
    """Summarise a batch of conversations using the OpenAI API."""

    # Create a new OpenAI client
    client = OpenAI()

    # List to collect all summaries for this batch
    out = []

    # Loop through each conversation text in the batch
    for conv_text in batch[text_key]:

        # If the conversation text is missing, NaN, or empty after stripping whitespace
        if conv_text is None or (isinstance(conv_text, float) and math.isnan(conv_text)) or str(conv_text).strip() == "":
            # Append an empty string as a placeholder summary
            out.append("")
            # Skip to the next conversation
            continue

        # Insert the conversation text into the prompt template
        prompt = prompt_template.format(conversation_data=str(conv_text))

        try:
            # Call the OpenAI Responses API with the given parameters
            resp = client.responses.create(
                model=model,                     # Which model to use
                input=prompt,                    # The formatted input prompt
                temperature=temperature,         # Sampling randomness
                max_output_tokens=max_output_tokens,  # Limit on output length
            )

            # Extract the text output, strip extra whitespace, and save it
            out.append(resp.output_text.strip())

        except Exception as e:
            # Print the error for debugging
            print(f"OpenAI error: {e}")
            # Stop immediately on the first error so it doesn't silently continue
            raise

    # Return the list of summaries as a dictionary
    return {"summary": out}

### Execution of Summarisation Loop

In [ ]:
# initialise progress bar (covers the entire run, not per batch)
total_to_do_int = int(len(ds_remaining))     # total number of rows left to summarise
pbar = tqdm(total=total_to_do_int, desc="Summarising", unit="rows")  # progress bar in rows
start_time = time.time()                     # record the start time for elapsed/ETA calculations
processed  = 0                               # counter for how many rows have been processed so far

# Loop through the dataset in steps of BATCH_SIZE
for start in range(0, total_to_do_int, BATCH_SIZE):
    end = min(start + BATCH_SIZE, total_to_do_int)  # ensure the last batch doesn’t overshoot
    # Select a slice of the dataset while keeping original row order (via orig_idx)
    batch = ds_remaining.select(range(start, end))

    # Run the summarisation function on this batch
    batch_done = batch.map(
        hf_openai_summarise,   # custom summarisation function
        batched=True,          # process rows in batches
        batch_size=BATCH_SIZE, # batch size for the map
        num_proc=NUM_PROC,     # parallelism level (set higher for more speed if API allows)
        fn_kwargs={            # parameters passed into hf_openai_summarise
            "model": MODEL,
            "temperature": TEMPERATURE,
            "max_output_tokens": MAX_OUTPUT_TOKENS,
            "prompt_template": SUMMARY_PROMPT,
            "text_key": text_col,
        },
        load_from_cache_file=False,  # disable cache
    )

    # Prepare batch for saving: convert to CSV-safe format and restore original order
    batch_out = normalize_for_csv(batch_done).sort("orig_idx")

    # Merge results with existing CSV if it already exists
    if os.path.exists(out_csv):
        # Load existing CSV into a dataset
        existing = load_dataset("csv", data_files=out_csv)["train"]
        # Concatenate existing results with the new batch, then sort by original index
        merged = concatenate_datasets([existing, batch_out]).sort("orig_idx")

        # Deduplicate by keeping the first row for each conversation_id
        seen = set()
        def keep_first(example):
            cid = example[id_col]
            if cid in seen:
                return False
            seen.add(cid)
            return True
        merged = merged.filter(keep_first, load_from_cache_file=False)

        # Write the merged dataset safely (temp file → replace)
        atomic_write_csv(merged, out_csv, tmp_csv)
    else:
        # If no CSV exists yet, just save this batch directly
        atomic_write_csv(batch_out, out_csv, tmp_csv)

    # ETA calculation
    processed = end                                     # update processed count
    elapsed = time.time() - start_time                  # time since start
    rate = processed / elapsed if elapsed > 0 else 0.0  # rows per second
    remaining = total_to_do_int - processed             # how many rows left
    eta = fmt_eta(remaining / rate) if rate > 0 else "unknown"  # estimate remaining time

    # progress bar update
    pbar.update(end - start)   # advance bar by number of rows in this batch
    pbar.set_postfix({"elapsed": fmt_eta(elapsed), "eta": eta})  # show elapsed + ETA on bar

    # Print detailed status line for logging
    print(f"Saved rows {start}–{end} | Processed={processed}/{total_to_do_int} | "
          f"Elapsed={fmt_eta(elapsed)} | ETA={eta}")

# Close the progress bar after all batches are processed
pbar.close()

Summarising:   6%|▌         | 100/1634 [01:18<20:02,  1.28rows/s, elapsed=00:01:18, eta=00:20:02]

Saved rows 0–100 | Processed=100/1634 | Elapsed=00:01:18 | ETA=00:20:02


Summarising:  12%|█▏        | 200/1634 [02:33<18:19,  1.30rows/s, elapsed=00:02:33, eta=00:18:22]

Saved rows 100–200 | Processed=200/1634 | Elapsed=00:02:33 | ETA=00:18:22


Summarising:  18%|█▊        | 300/1634 [03:49<16:54,  1.32rows/s, elapsed=00:03:49, eta=00:16:58]

...

Saved rows 1400–1500 | Processed=1500/1634 | Elapsed=00:19:44 | ETA=00:01:45


Summarising:  98%|█████████▊| 1600/1634 [21:12<00:29,  1.16rows/s, elapsed=00:21:12, eta=00:00:27]

Saved rows 1500–1600 | Processed=1600/1634 | Elapsed=00:21:12 | ETA=00:00:27


Summarising: 100%|██████████| 1634/1634 [21:44<00:00,  1.25rows/s, elapsed=00:21:44, eta=00:00:00]

Saved rows 1600–1634 | Processed=1634/1634 | Elapsed=00:21:44 | ETA=00:00:00


In [ ]:
from datasets import load_dataset, Features, Value
from huggingface_hub import login, HfApi

#saved CSV as a HF Dataset (no pandas)
out_csv = "/content/drive/MyDrive/fyp-2025/Datasets/Summaries/validation/history_summaries.csv"  #  path
ds = load_dataset("csv", data_files=out_csv)["train"]

# enforce column types and sort by original order
features = Features({
    "conversation_id": Value("string"),
    "history_string":   Value("string"),
    "summary":          Value("string"),
    "orig_idx":         Value("int64"),
})
ds = ds.cast(features)
ds = ds.sort("orig_idx")

#a repo and push the dataset
user_or_org = "Lakshan2003"                   # HF username/org
repo_name   = "customer_agent_validation_summaries" # name on the Hub
repo_id     = f"{user_or_org}/{repo_name}"

# Create the dataset repo if it doesn't exist (safe to run repeatedly)
api = HfApi()
api.create_repo(repo_id, repo_type="dataset", private=True, exist_ok=True)

# 5) Push to Hub (this uploads data + auto-creates a README stub)
ds.push_to_hub(repo_id, private=True)
print(f"Uploaded to https://huggingface.co/datasets/{repo_id}")

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :   0%|          | 49.1kB / 16.4MB            

Uploaded to https://huggingface.co/datasets/Lakshan2003/customer_agent_validation_summaries


### Test Dataset

In [ ]:
dataset_name   = "Lakshan2003/customer_service_200k_client_agent_conversations"
split          = "test"                 # which dataset split to use
text_col       = "history_string"        # column containing the conversation text
id_col         = "conversation_id"       # unique ID for each conversation

BATCH_SIZE     = 100                       # number of rows to process per batch
MAX_ROWS       = None                    # None → whole split; int → first N rows of the ORIGINAL dataset
NUM_PROC       = 2                        # number of processes for mapping (set >1 to speed up)

# Output paths
out_csv        = f"/content/drive/MyDrive/fyp-2025/Datasets/Summaries/{split}/history_summaries.csv"
tmp_csv        = out_csv + ".tmp"         # temp file for atomic writes

# OpenAI API settings
MODEL              = "gpt-4.1-nano"
TEMPERATURE        = 0.3
MAX_OUTPUT_TOKENS  = 250                   # ~90–140 words output

# CSV columns to keep in final output
COLUMNS_TO_SAVE = [id_col, text_col, "summary", "orig_idx"]

In [ ]:
# Load full dataset and tag each row with its original index for order preservation
ds_full = load_dataset(dataset_name)[split]
ds_full = ds_full.map(lambda ex, i: {"orig_idx": i}, with_indices=True, load_from_cache_file=False)

# If MAX_ROWS is set, only keep the first N rows while preserving order
if isinstance(MAX_ROWS, int):
    ds_target = ds_full.select(range(min(MAX_ROWS, len(ds_full))))
else:
    ds_target = ds_full

## Resume Logic
# Check if an output CSV already exists.
# If it does, read processed IDs and skip those rows to avoid duplicates.
if os.path.exists(out_csv):
    existing = load_dataset("csv", data_files=out_csv)["train"]
    done_ids = set(map(str, existing[id_col]))
    ds_remaining = ds_target.filter(lambda x: str(x[id_col]) not in done_ids)
else:
    # If no output CSV exists, start fresh with the full target set
    ds_remaining = ds_target

# Track how many rows are in the target window and how many remain
total_target = len(ds_target)
total_to_do  = len(ds_remaining)
print(f"[info] Target window: {total_target} rows (MAX_ROWS={MAX_ROWS if MAX_ROWS else 'ALL'})")
print(f"[info] Remaining: {total_to_do} rows to process")

# Stop execution if everything in the target set has already been processed
if total_to_do == 0:
    raise SystemExit("Nothing left to process in the selected window. All done.")
ds_full = load_dataset(dataset_name)[split]
ds_full = ds_full.map(lambda ex, i: {"orig_idx": i}, with_indices=True, load_from_cache_file=False)

[info] Target window: 36668 rows (MAX_ROWS=ALL)
[info] Remaining: 17268 rows to process


In [ ]:
# initialise progress bar (covers the entire run, not per batch)
total_to_do_int = int(len(ds_remaining))     # total number of rows left to summarise
pbar = tqdm(total=total_to_do_int, desc="Summarising", unit="rows")  # progress bar in rows
start_time = time.time()                     # record the start time for elapsed/ETA calculations
processed  = 0                               # counter for how many rows have been processed so far

# Loop through the dataset in steps of BATCH_SIZE
for start in range(0, total_to_do_int, BATCH_SIZE):
    end = min(start + BATCH_SIZE, total_to_do_int)  # ensure the last batch doesn’t overshoot
    # Select a slice of the dataset while keeping original row order (via orig_idx)
    batch = ds_remaining.select(range(start, end))

    # Run the summarisation function on this batch
    batch_done = batch.map(
        hf_openai_summarise,   # custom summarisation function
        batched=True,          # process rows in batches
        batch_size=BATCH_SIZE, # batch size for the map
        num_proc=NUM_PROC,     # parallelism level (set higher for more speed if API allows)
        fn_kwargs={            # parameters passed into hf_openai_summarise
            "model": MODEL,
            "temperature": TEMPERATURE,
            "max_output_tokens": MAX_OUTPUT_TOKENS,
            "prompt_template": SUMMARY_PROMPT,
            "text_key": text_col,
        },
        load_from_cache_file=False,  # disable cache
    )

    # Prepare batch for saving: convert to CSV-safe format and restore original order
    batch_out = normalize_for_csv(batch_done).sort("orig_idx")

    # Merge results with existing CSV if it already exists
    if os.path.exists(out_csv):
        # Load existing CSV into a dataset
        existing = load_dataset("csv", data_files=out_csv)["train"]
        # Concatenate existing results with the new batch, then sort by original index
        merged = concatenate_datasets([existing, batch_out]).sort("orig_idx")

        # Deduplicate by keeping the first row for each conversation_id
        seen = set()
        def keep_first(example):
            cid = example[id_col]
            if cid in seen:
                return False
            seen.add(cid)
            return True
        merged = merged.filter(keep_first, load_from_cache_file=False)

        # Write the merged dataset safely (temp file → replace)
        atomic_write_csv(merged, out_csv, tmp_csv)
    else:
        # If no CSV exists yet, just save this batch directly
        atomic_write_csv(batch_out, out_csv, tmp_csv)

    # ETA calculation
    processed = end                                     # update processed count
    elapsed = time.time() - start_time                  # time since start
    rate = processed / elapsed if elapsed > 0 else 0.0  # rows per second
    remaining = total_to_do_int - processed             # how many rows left
    eta = fmt_eta(remaining / rate) if rate > 0 else "unknown"  # estimate remaining time

    # progress bar update
    pbar.update(end - start)   # advance bar by number of rows in this batch
    pbar.set_postfix({"elapsed": fmt_eta(elapsed), "eta": eta})  # show elapsed + ETA on bar

    # Print detailed status line for logging
    print(f"Saved rows {start}–{end} | Processed={processed}/{total_to_do_int} | "
          f"Elapsed={fmt_eta(elapsed)} | ETA={eta}")

# Close the progress bar after all batches are processed
pbar.close()

Summarising:   1%|          | 100/17268 [01:33<4:26:45,  1.07rows/s, elapsed=00:01:33, eta=04:26:45]

Saved rows 0–100 | Processed=100/17268 | Elapsed=00:01:33 | ETA=04:26:45


Summarising:   1%|          | 200/17268 [03:15<4:40:05,  1.02rows/s, elapsed=00:03:15, eta=04:37:51]

Saved rows 100–200 | Processed=200/17268 | Elapsed=00:03:15 | ETA=04:37:51


Summarising:   2%|▏         | 300/17268 [04:46<4:28:41,  1.05rows/s, elapsed=00:04:46, eta=04:29:50]

...

Saved rows 17000–17100 | Processed=17100/17268 | Elapsed=05:58:52 | ETA=00:03:31


Summarising: 100%|█████████▉| 17200/17268 [6:00:31<01:40,  1.48s/rows, elapsed=06:00:31, eta=00:01:25]

Saved rows 17100–17200 | Processed=17200/17268 | Elapsed=06:00:31 | ETA=00:01:25


Summarising: 100%|██████████| 17268/17268 [6:02:11<00:00,  1.26s/rows, elapsed=06:02:11, eta=00:00:00]

Saved rows 17200–17268 | Processed=17268/17268 | Elapsed=06:02:11 | ETA=00:00:00


In [ ]:
from datasets import load_dataset, Features, Value
from huggingface_hub import login, HfApi

# 2) Load saved CSV as a HF Dataset (no pandas)
out_csv = "/content/drive/MyDrive/fyp-2025/Datasets/Summaries/test/history_summaries.csv"  #  path
ds = load_dataset("csv", data_files=out_csv)

# enforce column types and sort by original order
features = Features({
    "conversation_id": Value("string"),
    "history_string":   Value("string"),
    "summary":          Value("string"),
    "orig_idx":         Value("int64"),
})
ds = ds.cast(features)
ds = ds.sort("orig_idx")

# 4) Create (or reuse) a repo and push the dataset
user_or_org = "Lakshan2003"                   #  HF username/org
repo_name   = "customer_agent_test_summaries" # name on the Hub
repo_id     = f"{user_or_org}/{repo_name}"

# Create the dataset repo if it doesn't exist (safe to run repeatedly)
api = HfApi()
api.create_repo(repo_id, repo_type="dataset", private=True, exist_ok=True)

# 5) Push to Hub (this uploads data + auto-creates a README stub)
ds.push_to_hub(repo_id, private=True)
print(f"Uploaded to https://huggingface.co/datasets/{repo_id}")

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :   2%|1         |  528kB / 32.7MB            

Uploaded to https://huggingface.co/datasets/Lakshan2003/customer_agent_test_summaries


### Train Dataset

In [ ]:
dataset_name   = "Lakshan2003/customer_service_200k_client_agent_conversations"
split          = "train"                 # which dataset split to use
text_col       = "history_string"        # column containing the conversation text
id_col         = "conversation_id"       # unique ID for each conversation

BATCH_SIZE     = 100                       # number of rows to process per batch
MAX_ROWS       = None                    # None → whole split; int → first N rows of the ORIGINAL dataset
NUM_PROC       = 2                        # number of processes for mapping (set >1 to speed up)

# Output paths
out_csv        = f"/content/drive/MyDrive/fyp-2025/Datasets/Summaries/{split}/history_summaries.csv"
tmp_csv        = out_csv + ".tmp"         # temp file for atomic writes

# OpenAI API settings
MODEL              = "gpt-4.1-nano"
TEMPERATURE        = 0.3
MAX_OUTPUT_TOKENS  = 250                   # ~90–140 words output

# CSV columns to keep in final output
COLUMNS_TO_SAVE = [id_col, text_col, "summary", "orig_idx"]

In [ ]:
# Load full dataset and tag each row with its original index for order preservation
ds_full = load_dataset(dataset_name)[split]
ds_full = ds_full.map(lambda ex, i: {"orig_idx": i}, with_indices=True, load_from_cache_file=False)

# If MAX_ROWS is set, only keep the first N rows while preserving order
if isinstance(MAX_ROWS, int):
    ds_target = ds_full.select(range(min(MAX_ROWS, len(ds_full))))
else:
    ds_target = ds_full

## Resume Logic
# Check if an output CSV already exists.
# If it does, read processed IDs and skip those rows to avoid duplicates.
if os.path.exists(out_csv):
    existing = load_dataset("csv", data_files=out_csv)["train"]
    done_ids = set(map(str, existing[id_col]))
    ds_remaining = ds_target.filter(lambda x: str(x[id_col]) not in done_ids)
else:
    # If no output CSV exists, start fresh with the full target set
    ds_remaining = ds_target

# Track how many rows are in the target window and how many remain
total_target = len(ds_target)
total_to_do  = len(ds_remaining)
print(f"[info] Target window: {total_target} rows (MAX_ROWS={MAX_ROWS if MAX_ROWS else 'ALL'})")
print(f"[info] Remaining: {total_to_do} rows to process")

# Stop execution if everything in the target set has already been processed
if total_to_do == 0:
    raise SystemExit("Nothing left to process in the selected window. All done.")
ds_full = load_dataset(dataset_name)[split]
ds_full = ds_full.map(lambda ex, i: {"orig_idx": i}, with_indices=True, load_from_cache_file=False)

README.md:   0%|          | 0.00/834 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/171M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/24.3M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/48.7M [00:00<?, ?B/s]

[info] Target window: 128335 rows (MAX_ROWS=ALL)
[info] Remaining: 6235 rows to process


In [ ]:
# initialise progress bar (covers the entire run, not per batch)
total_to_do_int = int(len(ds_remaining))     # total number of rows left to summarise
pbar = tqdm(total=total_to_do_int, desc="Summarising", unit="rows")  # progress bar in rows
start_time = time.time()                     # record the start time for elapsed/ETA calculations
processed  = 0                               # counter for how many rows have been processed so far

# Loop through the dataset in steps of BATCH_SIZE
for start in range(0, total_to_do_int, BATCH_SIZE):
    end = min(start + BATCH_SIZE, total_to_do_int)  # ensure the last batch doesn’t overshoot
    # Select a slice of the dataset while keeping original row order (via orig_idx)
    batch = ds_remaining.select(range(start, end))

    # Run the summarisation function on this batch
    batch_done = batch.map(
        hf_openai_summarise,   # custom summarisation function
        batched=True,          # process rows in batches
        batch_size=BATCH_SIZE, # batch size for the map
        num_proc=NUM_PROC,     # parallelism level (set higher for more speed if API allows)
        fn_kwargs={            # parameters passed into hf_openai_summarise
            "model": MODEL,
            "temperature": TEMPERATURE,
            "max_output_tokens": MAX_OUTPUT_TOKENS,
            "prompt_template": SUMMARY_PROMPT,
            "text_key": text_col,
        },
        load_from_cache_file=False,  # disable cache
    )

    # Prepare batch for saving: convert to CSV-safe format and restore original order
    batch_out = normalize_for_csv(batch_done).sort("orig_idx")

    # Merge results with existing CSV if it already exists
    if os.path.exists(out_csv):
        # Load existing CSV into a dataset
        existing = load_dataset("csv", data_files=out_csv)["train"]
        # Concatenate existing results with the new batch, then sort by original index
        merged = concatenate_datasets([existing, batch_out]).sort("orig_idx")

        # Deduplicate by keeping the first row for each conversation_id
        seen = set()
        def keep_first(example):
            cid = example[id_col]
            if cid in seen:
                return False
            seen.add(cid)
            return True
        merged = merged.filter(keep_first, load_from_cache_file=False)

        # Write the merged dataset safely (temp file → replace)
        atomic_write_csv(merged, out_csv, tmp_csv)
    else:
        # If no CSV exists yet, just save this batch directly
        atomic_write_csv(batch_out, out_csv, tmp_csv)

    # ETA calculation
    processed = end                                     # update processed count
    elapsed = time.time() - start_time                  # time since start
    rate = processed / elapsed if elapsed > 0 else 0.0  # rows per second
    remaining = total_to_do_int - processed             # how many rows left
    eta = fmt_eta(remaining / rate) if rate > 0 else "unknown"  # estimate remaining time

    # progress bar update
    pbar.update(end - start)   # advance bar by number of rows in this batch
    pbar.set_postfix({"elapsed": fmt_eta(elapsed), "eta": eta})  # show elapsed + ETA on bar

    # Print detailed status line for logging
    print(f"Saved rows {start}–{end} | Processed={processed}/{total_to_do_int} | "
          f"Elapsed={fmt_eta(elapsed)} | ETA={eta}")

# Close the progress bar after all batches are processed
pbar.close()

Summarising:   2%|▏         | 100/6235 [01:49<1:51:55,  1.09s/rows, elapsed=00:01:49, eta=01:51:55]

Saved rows 0–100 | Processed=100/6235 | Elapsed=00:01:49 | ETA=01:51:55


Summarising:   3%|▎         | 200/6235 [03:46<1:54:17,  1.14s/rows, elapsed=00:03:46, eta=01:53:39]

Saved rows 100–200 | Processed=200/6235 | Elapsed=00:03:46 | ETA=01:53:39


Summarising:   5%|▍         | 300/6235 [06:00<2:01:36,  1.23s/rows, elapsed=00:06:00, eta=01:58:42]

...

Saved rows 6000–6100 | Processed=6100/6235 | Elapsed=02:17:46 | ETA=00:03:02


Summarising:  99%|█████████▉| 6200/6235 [2:19:48<00:46,  1.33s/rows, elapsed=02:19:47, eta=00:00:47]

Saved rows 6100–6200 | Processed=6200/6235 | Elapsed=02:19:47 | ETA=00:00:47


Summarising: 100%|██████████| 6235/6235 [2:20:43<00:00,  1.35s/rows, elapsed=02:20:43, eta=00:00:00]

Saved rows 6200–6235 | Processed=6235/6235 | Elapsed=02:20:43 | ETA=00:00:00


In [ ]:
from datasets import load_dataset, Features, Value
from huggingface_hub import login, HfApi

#saved CSV as a HF Dataset (no pandas)
out_csv = "/content/drive/MyDrive/fyp-2025/Datasets/Summaries/train/history_summaries.csv"  #  path
ds = load_dataset("csv", data_files=out_csv)["train"]

# enforce column types and sort by original order
features = Features({
    "conversation_id": Value("string"),
    "history_string":   Value("string"),
    "summary":          Value("string"),
    "orig_idx":         Value("int64"),
})
ds = ds.cast(features)
ds = ds.sort("orig_idx")

#a repo and push the dataset
user_or_org = "Lakshan2003"                   # HF username/org
repo_name   = "customer_agent_train_summaries" # name on the Hub
repo_id     = f"{user_or_org}/{repo_name}"

# Create the dataset repo if it doesn't exist (safe to run repeatedly)
api = HfApi()
api.create_repo(repo_id, repo_type="dataset", private=True, exist_ok=True)

# 5) Push to Hub (this uploads data + auto-creates a README stub)
ds.push_to_hub(repo_id, private=True)
print(f"Uploaded to https://huggingface.co/datasets/{repo_id}")

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  15%|#4        | 16.8MB /  114MB            

Uploaded to https://huggingface.co/datasets/Lakshan2003/customer_agent_train_summaries


### Merge Datasets

In [ ]:
from datasets import load_dataset, concatenate_datasets

# Load base dataset
base = load_dataset("Lakshan2003/customer_service_200k_client_agent_conversations")

# Load summaries and combine
train_sum = load_dataset("Lakshan2003/customer_agent_train_summaries")["train"]
test_sum  = load_dataset("Lakshan2003/customer_agent_test_summaries")["train"]
val_sum   = load_dataset("Lakshan2003/customer_agent_validation_summaries")["train"]

all_summaries = concatenate_datasets([train_sum, test_sum, val_sum])
all_summaries = all_summaries.remove_columns(
    [col for col in all_summaries.column_names if col not in ["conversation_id", "summary"]]
)

# Build a fast lookup dict {conversation_id: summary}
summary_lookup = {row["conversation_id"]: row["summary"] for row in all_summaries}

# Flatten base dataset
all_base = concatenate_datasets([base["train"], base["test"], base["validation"]])
cols_to_remove = [c for c in ["history_token_count"] if c in all_base.column_names]
all_base = all_base.remove_columns(cols_to_remove)

# Map summaries using dictionary lookup
merged = all_base.map(
    lambda ex: {"summary": summary_lookup.get(ex["conversation_id"], None)}
)

Map:   0%|          | 0/183337 [00:00<?, ? examples/s]

In [ ]:
merged

Dataset({
    features: ['conversation_id', 'instruction', 'client_question', 'agent_answer', 'history', 'history_string', 'summary'],
    num_rows: 183337
})

In [ ]:
merged[0]

{'conversation_id': '8b3cfca253084f7284c3d5b618843980',
 'instruction': 'You are a professional customer service agent working at **Crest Financial Partners**, a financial services institution. Carefully review the full conversation history between the client and agent, along with any additional context provided. Use all available information to generate a clear, helpful, and professional agent response representing **Crest Financial Partners**.',
 'client_question': "Nope, that's all. Thanks so much, Dena!",
 'agent_answer': "You're welcome, Shirley. Have a great day!",
 'history': [{'speaker': 'agent',
   'text': 'Good morning, thank you for calling Crest Financial Partners. My name is Dena, how can I assist you today?'},
  {'speaker': 'agent',
   'text': 'Of course, Shirley. Can you please verify your identity for me by providing your full name and date of birth?'},
  {'speaker': 'client',
   'text': 'Sure, my name is Shirley Johnson and my date of birth is February 12,, 1980.'},
  

In [ ]:
merged.push_to_hub("Lakshan2003/customer_service_200k_with_summaries", private=True)

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/92 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  13%|#3        | 19.4MB /  149MB            

Creating parquet from Arrow format:   0%|          | 0/92 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :   5%|4         | 6.82MB /  148MB            

CommitInfo(commit_url='https://huggingface.co/datasets/Lakshan2003/customer_service_200k_with_summaries/commit/692cb6b6dd692a98bb717788cc855092638b46a2', commit_message='Upload dataset', commit_description='', oid='692cb6b6dd692a98bb717788cc855092638b46a2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Lakshan2003/customer_service_200k_with_summaries', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Lakshan2003/customer_service_200k_with_summaries'), pr_revision=None, pr_num=None)